# Функции (2 занятие)

In [ ]:
def foo(a, b = 42):
    return a + b

# был передан параметр по умолчанию
foo(42)

84

#### О чем надо помнить про параметры по умолчанию

Сначала должны идти обязательные параметры, и только потом необязательные (со значением по умолчанию)

In [1]:
# Вот так делать нельзя
def foo(a = 4, b):
    ...

SyntaxError: non-default argument follows default argument (4215674538.py, line 2)

### Первая задачка на добавление элементов в массив по умолчанию

In [15]:
def append_to_list(what, to=[]):
    to.append(what)
    return to

a = []
append_to_list(42, a)
append_to_list(43, a)

# вот такая проблемка может быть
append_to_list(42), append_to_list(42), append_to_list(42)


([42, 42, 42], [42, 42, 42], [42, 42, 42])

Разберем по порядку то, почему такая проблема возникает. Это связано в первую очередь со ссылками и то, как питон создает новые объекты

```
def append_to_list(what, to=[]):
```

Тут в параметре создается новый объект-список `to=[]` которые создается всего 1 раз на момент когда питон читает и выгружает эту функции в память. Поэтому когда мы вызовем функцию неправильно, то есть не передадим вторым параметром массив при вызове функции, то есть сделаем так:

```
append_to_list(42)
append_to_list(42)
```

то по итогу мы каждый раз будем добавлять элемент в один и тот же список `to`, который мы когда-то объявили в памяти

Еще хуже можно сделать при вызоыве функции в строку через запятую, в таком случае создается кортеж (tuple), это неизменяемый аналог обычного списка, по итогу при вызове первой функции она вернет [42], при вызове второй [42, 42], и потом так как эти фукнции возвращают не значение, а ссылку на один и тот же объект в памяти, то есть массив `to=[]`, у нас оно соберется в один двумерный массив

Аналог операции в строку выглядит вот так:

In [14]:
res1 = append_to_list(42)
res2 = append_to_list(42)
res3 = append_to_list(42)

res = [res1, res2, res3]
res

[[42, 42, 42], [42, 42, 42], [42, 42, 42]]

Эта проблема может возникать со всем, что является ссылкой

Дефолтные параметры объекта можно узнать следующим образом

In [16]:
append_to_list.__defaults__

([42, 42, 42],)

Ну и как видно, тут параметр по дефолту уже не чистый массив, а загрязненный

Первый фикс данной проблемы заключается в том, что мы будем возвращать не ссылку на один и тот же массив, а будем спред оператором возвращать копию массива

In [20]:
def append_to_list(what, to=[]):
    return [*to, what]

a = []
# указатель объекта
# ids = [id(a)]

# for i in range(5):
#     a = append_to_list(42, a)
#     ids.append(id(a))

# ids
append_to_list(43)
append_to_list(43)


[43]

In [24]:
def append_to_list(what, to=[]):
    return [*to, what]

a = []
# функция id позволяет получить указатель объекта
ids = []

for i in range(5):
    a = append_to_list(42, a)
    ids.append(a)

ids
# append_to_list(43)
# append_to_list(43)


[[42], [42, 42], [42, 42, 42], [42, 42, 42, 42], [42, 42, 42, 42, 42]]

По результату видно, что получаются ращные списки, для наглядности выведем их ссылки с помощью функции `id()`

In [26]:
def append_to_list(what, to=[]):
    return [*to, what]

a = []
ids = [id(a)]
a

[]

In [27]:
for i in range(5):
    a = append_to_list(42, a)
    ids.append(id(a))

ids

[4418119360, 4418445632, 4417709504, 4418445632, 4417709504, 4418445632]

Ну и по ссылкам на объекты видно, что все списки разные

Из возможных гипотетических проблем данной реализации:
- могут совпасть адреса в памяти, если старый объект был удален сборщиком мусора, но не в данном случае так как все списки сохранены в массив ids

Но главная проблема следующая:
- В данной реализации сигнатура функции обещает добавить элемент в массив, а по факту возвращает новый список со старыми элементами и новым добавленым, а не просто добавляет в список, и это очень плохо

Поэтому можно решить задачу следующим, эталонным способом:

In [ ]:
a = []
ids = [id(a)]

def append_to_list(what, to=[]):
    if to is None:
        to = []
    to.append(what)
    return to

for i in range(5):
    a = append_to_list(i, a)
    ids.append(id(a))

ids

[4418217408, 4418217408, 4418217408, 4418217408, 4418217408, 4418217408]

По айдишникам видно, что мы честно добавляем элемент в один и тот же список, потому что они одинаковые

In [30]:
a

[0, 1, 2, 3, 4]

Значения тоже все верные

Есть еще пример более запутанной реализации данной функции, который обычно не рекомендуют использовать из-за сложности чтения:

In [ ]:
a = []

ids = [id(a)]

def append_to_list(what, to=None):
    # негодяй
    to = to or []

    to.append(what)

    return to

for i in range(5):
    a = append_to_list(i, a)
    ids.append(id(a))

ids

[4415837888, 4391416000, 4391416000, 4391416000, 4391416000, 4391416000]

В таком случае у нас почему-то сслыка на первый элемент отличается от всех остальных??

Это происходит из-за того, как питон сравнивает объекты%

На первой итерации to=[] пустому массиву, и в логическом сравнении пустой массив = False

То есть левая часть равна False, так как массив пустой, и возвращается новый массив, и полностью игнорирует переданный пустой массив

То есть поведение на пустой переданный массив и не переданный массив вообще в аргументы одинаковые, что как бы не правильно

Дальше мы уже передаем массив с 1 элементом и проверка to возращает его, а это массив с другим адресом


Исправляется это дело доп проверкой и по факту мы возвращаемся к предыдущему решению

In [32]:
a = []

ids = [id(a)]

def append_to_list(what, to=None):
    to = [] if to is None else to

    to.append(what)
    return to

for i in range(5):
    a = append_to_list(i, a)
    ids.append(id(a))

ids

[4417692160, 4417692160, 4417692160, 4417692160, 4417692160, 4417692160]

## Идеалогия питона

In [33]:
# основный идеи питона
import this

The Zen of Python, by Tim Peters

Beautiful is better than ugly.
Explicit is better than implicit.
Simple is better than complex.
Complex is better than complicated.
Flat is better than nested.
Sparse is better than dense.
Readability counts.
Special cases aren't special enough to break the rules.
Although practicality beats purity.
Errors should never pass silently.
Unless explicitly silenced.
In the face of ambiguity, refuse the temptation to guess.
There should be one-- and preferably only one --obvious way to do it.
Although that way may not be obvious at first unless you're Dutch.
Now is better than never.
Although never is often better than *right* now.
If the implementation is hard to explain, it's a bad idea.
If the implementation is easy to explain, it may be a good idea.
Namespaces are one honking great idea -- let's do more of those!


## Лямбда функция (анонимные функции)

В питоне по идеалогии сами разработчики рекомендуется не использовать лямбда функции

Разработчики специально в объявление анонимной функции добавили слово лямбда, чтобы отбить желание ее использовать, в других языках анонимнве функции например по другому выглядят

In [ ]:
mult_l = lambda a, b: a + b

<function __main__.<lambda>(a, b)>

Под капотом лямбда функция это такая же фукнция как и через `def` объявленная через объект `PyFunctionObject`, за исключением того, что в поле `__name__` будет написано слово `lambda`

Лямбда функция может принимат любое кол-во параметров, но выполнит всего одно выражение, сколь угодно длинное

#### в питоне знак = (присвоить) находится на самом верхнем уровне операций

Также = - это инструкция, как и if, for, def

А операторы - это + - * and и тд

## Позиционные и не позиционные аргументы

`*` - одна звезда говорит питону о том, что нужно собрать все позиционные параметры в эту сторону
`**` - две звезды говорят о том, что нужно все непозиционные аргументы собрать в другую сторону

In [ ]:
def foo(*a, **b):
    print(a, b)

foo(1, 2, 3, a=4, b=5, d = 6)
# (1, 2, 3) {'a': 4, 'b': 5, 'd': 6}

(1, 2, 3) {'a': 4, 'b': 5, 'd': 6}


In [ ]:
# можно через звезду разделить эти элементы, но тогда мы сможем только 1 передать, а не список
def foo(a, *, b = 4):
    print(a, b)

foo([1, 2, 3], b = 4)

[1, 2, 3] 4


In [40]:
def foo(*a, d, **b):
    print(a, b)

foo(1, 2, 3, d = 6, a=4, b=5)
# d не выведется, потому что он является параметром по ключу, а не позиционным или непозиционным

(1, 2, 3) {'a': 4, 'b': 5}


In [40]:
# Все возможные способы передачи параметров в современном питоне

# до / - все позиционные аргументы, без имени
# между / и * - любые аргументы
# после * - только именнованные аргументы
# **kwargs - сборка всех послеидущих именованных аргументов

def foo(a, b, /, c, d, *, e, f, **kwargs):
    print(a, b, c, d, e, f, kwargs)

foo([1, 2, 3], 4, 3, d = 4, e = 3, g = 4, f = 4, gg = 15, dd = 33 )

[1, 2, 3] 4 3 4 3 4 {'g': 4, 'gg': 15, 'dd': 33}


Почти никто так никогда не делает, можно сделать когда функцию плохо читать и мы хотим чтобы код выглядел как дифферинцииальные уравнения

## Функция, которая складывает все числа слева

In [ ]:
# функция которая типа как факториал работает и складывает или делает действие с массиврм

# reduce - аналог такой функции, она аккумулирует значение по операции, которую мы передали, то есть мы перемножили числа
import operator

def left_fold(*values, operator):
    value = values[0]

    for i in range(1, len(values)):
        value = operator(value, values[i])
    return value

left_fold(1, 2, 3, 4, 5, operator=operator.mul)

120

Аннотации - они по факту просто для сигнатуры нужны, но питон сам по себе не проверяет и нужна для сигнатуры функции, то есть это просто документация

In [45]:
def foo(a: int, b: int) -> int:
    """Какая то ультра крутая аннотация для функции"""
    return a + b

In [48]:
foo.__annotations__

{'a': int, 'b': int, 'return': int}

In [47]:
foo.__name__

'foo'

In [49]:
foo.__doc__

'Какая то ультра крутая аннотация для функции'

In [ ]:
a : str = '42' # но так обычно не делают, потому что мы и так знаем что это строка

В аннотациях типов параметров что угодно может лежать и питон это пропустит

In [50]:
def foo(a: 1 + 2 + 3, b: [4 ,5 ,6]) -> int:
    return a + b

foo('a', 'b')

'ab'

У аннотации есть доп инструменты

In [ ]:
import typing

def foo(a: typing.Union[int, float] | typing.Literal[456]) -> int | float:
    return a

foo(43)

help(foo)

Help on function foo in module __main__:

foo(a: Union[int, float, Literal[456]]) -> int | float



Тип по больше части нужны для удобства написания кода, так как 
Статические проверки - ide
Есть такие которые на каждый вызов срабатывают

In [ ]:
foo = lambda a, b: a + b
foo(1, 2)

# лямбда может выполнить всего 1 операцию, специально добавили слово lambda чтобы разработчики не использовали эту функцию
# сюда нельзя писать аннотации

In [ ]:
foo.__annotations__ = {'a' : int, 'b' : int}
foo.__defaults__ =(10,)

foo(1)

11

В питоне нет магии, то есть любую функцию которая используется в питоне, омжно на самом питоне написать

Короче лямбдами не пользуемся

#### Замыкания

Это функция, которая помнит свое лексическое окружение и имеет доступ к переменным из внешней области видимости

In [38]:
#  функция в функции

def foo(a, b):
    def bar(c, d):
        return c + d

    return a * bar(a, b) + b


foo(1, 2)
# это позволяет реализовать замыкания - запомнили функцией те значения функции которые были в пространстве имен этой функции

5

In [ ]:
def foo(a, b):
    def bar(c, d):
        return b * bar(a, b) + c

    a = 10
    return bar

adds = []

for i in range(10):
    adds.append(lambda a: a + i)

adds[9](2)

# все функции запомнили одну и ту же лексическую область i и теперь ссылаются на нее в памяти, то есть равны все 9


11

Эту пробему можно решить следующим образом - для внешней переменно i которая затемняет значение внутренней i создается свое новое лексическое окружение, происходит замыкание и она грубо говоря пересоздается внутри этого лексического окружения, за счет чего является другой i и содержит другое значение

In [ ]:
adds = []

for i in range(10):
    adds.append((lambda i: (lambda a: a + i))(i))

adds[8](1)

9

Ну и можно тоже самое сделать не через анонимную функцию, а через обычную

In [ ]:
def make_incrementer(i):
    return lambda a: a + i

adds = []

for i in range(10):
    adds.append(make_incrementer(i))
adds[9](1)

# то есть можно вот так улучшить функцию

10

### nonlocal and global

In [ ]:
def foo(a, b):
    def bar(c):
        # из-за порядка объявления питон считает a локальной переменной и в return вызовет ошибку, так как на уровне выше ее уже не существует
        a += c

    for i in range(b):
        bar(i)

    return a

foo(0, 5)

UnboundLocalError: local variable 'a' referenced before assignment

В питоне такой код не сработает потому что a всегда меняется a внутри своей функции, а выше какой был сверху

Надо использовать nonlocal - доступ к ближайшей верхнему лексическому окружению, нде есть эта переменная

In [ ]:
def foo(a, b):
    def bar(c):
        nonlocal a
        a += c

    for i in range(b):
        bar(i)

    return a

foo(0, 5)

Если переменной нет то вызовется ошибка

In [ ]:
def foo(a, b):
    def bar(c):
        global a
        a += c

    for i in range(b):
        bar(i)

    return a

foo(0, 5)

# global говорит что надо пойти на верхний уровень и сразу искать где то там переменнецю a, а не по дереву вызовов
# не надо пользоваться такими операторами

### Создание динамического атрибута у объекта функции

В питоне функции являются объектами первого порядка, как строки списки и словари, поэтому у них могут свои. личные свойства (атрибуты), которые можно прям на ходу создавать через точку

In [2]:
# для таких ситуация в питоне можно сделать замыкание по имени функции

def foo(a, b):
    def bar(c):
        bar.a += c

    bar.a = a

    for i in range(b):
        bar(i)

    # вместо а возвращает а из bar, потому что а глобальные а не поменялось
    return bar.a

foo(0, 5)

# перменную внутри функции можно прописать любую таким способом, и получается так писать

10

Еше способ для того, чтобы получить доступ к значению из другой области видимости

Тут стоит вспомнить о том, что есть изменяемые и неизменяемые данные в питоне

1) Неизменяемые - менять после создания нельзя, при попытке изменить создается другая ячейка в памяти
- Числа (int, float, complex)
- Строки
- Кортеж (tuple) (1, 2, 3)
- Логический тип (bool)
- frozenset

2) Изменяемые - такие объекты можно спокойно редактировать
- Список (list) [1, 2, 3]
- Словари (dict) {"key" : "value"}
- Множества (set) - {1, 2, 3}
- Массив байт (bytearray)

Собственно, следующая идея заключается в том, чтобы неизменямый тип привести к изменяемому, за счет этого объект не будет пересоздаваться а мы просто получим ссылку на существующий

In [8]:
# можно завести список, потому что состояние списка можно менять, в отличие от числа, так тоже писать надо стараться избегать
def foo(a, b):
    
    a = [a] 
    print(a)

    def bar(c):
        a[0] += c

    for i in range(b):
        print(a)

        bar(i)

    return a[0]

foo(1, 5)

[1]
[1]
[1]
[2]
[4]
[7]


11

In [22]:
def create_btn():
    clicks = 0 

    def click():
        nonlocal clicks
        clicks += 1
        print(f"Клик! Всего: {clicks}")

    return click

# надо вызывать через инстанс, потому что только так значение счетчика сохранится в памяти
btn = create_btn()
btn()
btn()


Клик! Всего: 1
Клик! Всего: 2


In [26]:
def create_btn():
    clicks = 0 
    clicks = [clicks]


    def click():
        clicks[0] += 1
        print(f"Клик! Всего: {clicks}")

    return click

# надо вызывать через инстанс, потому что только так значение счетчика сохранится в памяти
btn = create_btn()
btn()
btn()


Клик! Всего: [1]
Клик! Всего: [2]


`Callable[[аргументы], возвращаемое значение]`

`...` - любой тип

### Распаковка для последовательности

In [160]:
def foo(a, b):
    return a + b

foo(*[1, 2])

3

In [166]:
print(*[1, 2])

1 2


In [168]:
print(*{'1' : 2, '2' : 3})

1 2


### Распаковка для словаря

Нужна для распаковки словаря в пары (ключ=значение)

Используется для передаачи в функции именованных аргументов

In [163]:
def foo(a, b):
    return a + b

foo(**{'b' : 2, 'a' : 3})
foo(*[3], **{'b': 2})

5

Более наглядный пример, где распаковка словаря используется вместо передачи значений вручную

In [169]:
def setup_profile(name, age, city):
    print(f"{name}, {age}, {city}")

user_data = {
    'name' : 'ksenia',
    'age' : 22,
    'city' : 'VDK'
}

setup_profile(**user_data)

ksenia, 22, VDK


В питоне нет интерпретации в jump поэтому код фукнции вставляется прям там где он вызывается, а не перескакивает в памяти, из-за этого код чуть медленнее работает, но в срденем никакого замедления нет так как язык сам по себе медленный

Почти все компилируемые языки ставит джампы (возможно наоборот)

In [123]:
# почитаем фибоначи - текущее = сумма предыдущих 

def fibbonacci(n: int) -> int:
    # можно match добавить типа свитч кейс

    match n:
        case 0:
            return 0
        case 1:
            return 1
        case _:
            return fibbonacci(n - 2) + fibbonacci(n - 1)

    # if n == 0:
    #     return 0
    # if n == 1:
    #     return 1
    
    # return fibbonacci(n - 2) + fibbonacci(n - 1)

fibbonacci(4)

3

In [127]:
# почитаем фибоначи - текущее = сумма предыдущих 

def fibbonacci(n: int) -> int:
    # можно match добавить типа свитч кейс

    a, b = 0, 1
    for i in range(n):
        a, b = b, a + b

    return a

fibbonacci(5)

5

In [ ]:
# одномерная динамика

# почитаем фибоначи - текущее = сумма предыдущих 

def fibbonacci(n: int) -> int:
    # можно match добавить типа свитч кейс

    values = 

    match n:
        case 0:
            return 0
        case 1:
            return 1
        case _:
            return fibbonacci(n - 2) + fibbonacci(n - 1)

    # if n == 0:
    #     return 0
    # if n == 1:
    #     return 1
    
    # return fibbonacci(n - 2) + fibbonacci(n - 1)

fibbonacci(4)

# надо пинаписать декоратор, который помнит какие значения уже почитаны

### Практика 2

In [ ]:
def Add(x, y):
    return x + y

In [31]:
def PrintMatrix(matrix: list[list[int]]):
    for el in matrix:
        print(*el)

PrintMatrix([[1, 2, 3], [4, 5, 6]])

1 2 3
4 5 6


In [43]:
from typing import Callable

def Map(func: Callable, l: list):
    for i in range(len(l)):
        l[i] = func(l[i])

    return l

print(sum(Map(lambda x: x**2, [2, 3])))


13


In [ ]:
def Join(array, separator = ' '):
    resstr = ''
    for i in range(len(array)):
        resstr += array[i]

        if i == len(array) - 1:
            break

        resstr += separator

    return resstr
    
ary = ['aa', 'bb', 'cc']
print(Join(ary, '!'))




aa!bb!cc


Что тут надо помнить

Оператор AND - возвраащет первое false значение в цепочке, если все истина, то последнее

Оператор OR - возвращает первое true значение в цепочке, если все ложно то последнее

In [ ]:
def processingForDict(data: dict, key: str, kind : str | None = None):
    return (key in data and (kind is None or data.get("kind") == kind)) and data[key]


def processing(data: dict, kind: str | None = None, counter = None):
    counter = counter or 0

    is_str = type(data) is str
    is_dict = type(data) is dict
    is_list = type(data) is list

    key = (is_dict and next(iter(data))) or None

    current_kind = (kind or (is_dict and 'kind' in data and data['kind']) or key) or None

    print('Данные', data)
    print('Ключ из объекта', key)
    print('итоговый ключ', current_kind)
    print('-----')

    return (
        (
         (is_dict and processingForDict(data, "name", kind)) and processing(data['name'], current_kind) or
         (is_dict and processingForDict(data, "names", kind)) and processing(data['names'], current_kind) or
         (is_str and [data]) or []) 
        + 
        (is_list and counter < len(data) and (
            processing(data[counter], current_kind, 0) +
            processing(data, kind, counter + 1)
        ) or (
            is_dict and (key is not False) and processing(data[key], current_kind)
        ) or (
            is_dict and (key is False) and processing(data.values(), current_kind)
        ) or [])
    )

    # ) or (
    #     is_str and "name" in data and processingForStr(data["name"])
    # )

    
    
    # or (type(data[key]) is dict and key == "name" and (routergForDict(data, kind), True)) or (type(data[key]) is list and key == "name" and (routergForDict(data, kind), True)) or (type(data[key]) is list and key != "name" and (processingForCollection(data[key], kind), True))


def get_all_names(data: str | dict | list, kind: str | None = None) -> list[str]:


    
    """Collect all names from the `data` object. If `kind` is not None, only collects names of that kind.

    Arguments:
        data: object to collect names from

    Returns:
        Collected names
    """

    
    return (type(data) is str and processingForStr(data, kind)) or (type(data) is dict and routergForDict(data, kind)[0])

data2 = {
    'human': [
{
                'name': 'lyonya',
                'kind': 'animal'
            },
            {
                'name': 'vasya',
            },
            
        ]
} 

data3 = {
    'animal': [
{
                'name': 'ksenia',
                'kind': 'human'
            },
            {
                'name': 'lion',
            },
            
        ]
} 

data1 =  {
    'name': 'petya',
    'kind': 'human'
}

	

data = [
    'petya',
    {
        'names': ['barsik', 'mursik'],
        'kind': 'animal',
        'garbage': 123
    },
    {
        'human': [
            {
                'name': 'vasya',
            },
            {
                'name': 'lyonya',
                'kind': 'animal'
            }
        ]
    }
]


# print(get_all_names('fdsf'))

print((processing([data])))

petya barsik mursik ;
petya ;
petya barsik mursik ;


Если ключ false - то текущий объект это список []

Если ключ True - то текущий объект это объект {}, строка

data2 = {
    'human': [
        {
        'human': [
                {
                    'name': 'vasya',
                },
                'human': [
                    {
                    'human': [
                            {
                                'name': 'vasya',
                            },
                            {
                                'name': 'lyonya',
                                'kind': 'animal'
                            }
                        ]
                },
            ]
    },

            {
            'name': 'lyonya',
            'kind': 'animal'
            }
    ]
} 

1 рекурсия:

Если мы видм ключ, то заходим внутрь объекта рекурсивно

def f(data: dict):
    key = key(iter(data)) or false

    return (key and f(data[key]))

    

In [ ]:
def get_all_names(data: str | dict | list, kind: str | None = None) -> list[str]:
    """Collect all names from the `data` object. If `kind` is not None, only collects names of that kind.

    Arguments:
        data: object to collect names from

    Returns:
        Collected names
    """
    accum = []

    _process(data, accum, kind)

    return accum


def _process(data, accum: list[str], kind: str | None = None, counter: int = 0):
    is_str = type(data) is str
    is_dict = type(data) is dict
    is_list = type(data) is list

    new_kind = (is_dict and data.get('kind')) or kind

    is_str and (kind is None or kind == new_kind) and accum.append(data)

    is_list and counter < len(data) and (
        (_process(data[counter], accum, kind)) or
        _process(data, accum, kind, counter + 1)
    )

    is_dict and 'name' in data and (
        _process(data['name'], accum, new_kind)
    )

    is_dict and 'names' in data and (
        _process(data['names'], accum, new_kind)
    )

    is_dict and 'name' not in data and 'names' not in data and (
        _process(list(data.values()), accum, new_kind)
    )
	
data = [
    123,
    {
        'names': ['barsik', 'mursik'],
        'kind': 'animal',
    },
    {
        'names': ['petya', 'vasya', 123],
        'kind': 'human',
    }
]

print(((get_all_names(data, 'human'))))

['barsik', 'mursik', 'petya', 'vasya']


In [255]:
def get_all_names(data: str | dict | list, kind: str | None = None) -> list[str]:
    """Collect all names from the `data` object. If `kind` is not None, only collects names of that kind.

    Arguments:
        data: object to collect names from

    Returns:
        Collected names
    """
    accum = []

    _process(data, accum, kind)

    return accum


def _process(data, accum: list[str], kind: str | None = None, counter: int = 0, is_inner: bool = False):
    is_str = type(data) is str
    is_dict = type(data) is dict
    is_list = type(data) is list

    is_str and accum.append(data)

    # current_kind = (
    # (is_dict and data.get('kind')) or
    # (is_dict and 'name' not in data and 'names' not in data and list(data.keys())[0]) or
    # None
    # )

    current_kind = (is_dict and data.get('kind')) or None

    is_valid = (current_kind is kind) or (kind is None)
    is_inner_valid = is_inner and ((current_kind is kind) or current_kind is None)

    # print('curretn', current_kind)
    # print('parent', kind)
    # print('data', data)
    # print('is_valid', is_valid)
    # print('is_inner_valid', is_inner_valid)
    # print('counter', counter)
    # print('is_inner', is_inner)

    # print('\n')

    not is_inner and (is_dict and 'name' in data and is_valid and (_process(data['name'], accum, kind)) 
                   or is_dict and 'names' in data
    and is_valid and (
                        _process(data['names'], accum, kind))
    or is_list and counter < len(data) and (
                        (_process(data[counter], accum, kind)) or
                        _process(data, accum, kind, counter + 1))
    or (
        (is_dict and kind in data ) and _process(data[kind], accum, kind, counter=0, is_inner=True)
    )) or is_inner and (is_dict and 'name' in data and is_inner_valid and  (
                            _process(data['name'], accum, kind, is_inner=True)
        ) or is_dict and 'names' in data and is_inner_valid and (
                            _process(data['names'], accum, kind, is_inner=True))
        or is_list and counter < len(data) and (
                            (_process(data[counter], accum, kind, is_inner=True)) or
                            _process(data, accum, kind, counter + 1, is_inner=True)
        )
        or (
            (is_dict and kind in data) and _process(data[kind], accum, kind, is_inner=True)
        ))

data = [
    123,
    {
        'names': ['barsik', 'mursik'],
        'kind': 'animal',
    },
    {
        'names': ['petya', 'vasya', 123],
        'kind': 'human',
    }
]

print(*get_all_names(data), ';')
print(*get_all_names(data, 'human'), ';')
print(*get_all_names(data, 'animal'), ';')


barsik mursik petya vasya ;
petya vasya ;
barsik mursik ;


In [267]:
def is_str(value):
    return type(value) is int

all(map(lambda x: is_str(x), [1, 2, '4']))

False

In [266]:
(all(map(lambda x: is_str(x), [1, 2, '4'])))

False

In [269]:
import builtins


def get_all_names(data: str | dict | list, kind: str | None = None) -> list[str]:
    """Collect all names from the `data` object. If `kind` is not None, only collects names of that kind.

    Arguments:
        data: object to collect names from

    Returns:
        Collected names
    """
    accum = []

    _process(data, accum, kind)

    return accum

def is_str(value):
    return type(value) is str

def _process(data, accum: list[str], kind: str | None = None, counter: int = 0, is_inner: bool = False):
    is_str = type(data) is str
    is_dict = type(data) is dict
    is_list = type(data) is list

    is_str and accum.append(data)

    # current_kind = (
    # (is_dict and data.get('kind')) or
    # (is_dict and 'name' not in data and 'names' not in data and list(data.keys())[0]) or
    # None
    # )

    current_kind = (is_dict and data.get('kind')) or None

    print('curretn', current_kind)
    print('parent', kind)
    print('data', data)
    print('condition', is_dict and all(map(lambda x: is_str(x), data['names'])))

    print('\n')

    is_valid = (current_kind is kind) or (kind is None)
    is_inner_valid = is_inner and (current_kind is kind) or current_kind is None

    not is_inner and (is_dict and 'name' in data and is_valid and (_process(data['name'], accum, kind)) 
                   or is_dict and 'names' in data and builtins.all(map(lambda x: is_str(x), data['names']))
    and is_valid and (
                        _process(data['names'], accum, kind))
    or is_list and counter < len(data) and (
                        (_process(data[counter], accum, kind)) or
                        _process(data, accum, kind, counter + 1))
    or (
        (is_dict and kind in data ) and _process(data[kind], accum, kind, is_inner=True)
    )) or is_inner and (is_dict and 'name' in data and is_inner_valid and  (
                            _process(data['name'], accum, kind)
        ) or is_dict and 'names' in data and is_inner_valid and builtins.all(map(lambda x: is_str(x), data['names'])) and (
                            _process(data['names'], accum, kind)
        or is_list and counter < len(data) and (
                            (_process(data[counter], accum, kind)) or
                            _process(data, accum, kind, counter + 1)
        ))
        or (
            (is_dict and kind in data) and _process(data[kind], accum, kind, is_inner=True)
        ))

data = [
    'petya',
    {
        'names': ['barsik', 'mursik'],
        'kind': 'animal',
        'garbage': 123
    },
    # {
    #     'human': [
    #         {
    #             'name': 'vasya',
    #         },
    #         {
    #             'name': 'lyonya',
    #             'kind': 'animal'
    #         }
    #     ]
    # }
]


# ['petya', 'barsik', 'mursik', 'vasya', 'lyonya']

print(((get_all_names(data, 'human'))))

curretn None
parent human
data ['petya', {'names': ['barsik', 'mursik'], 'kind': 'animal', 'garbage': 123}]
condition False


curretn None
parent human
data petya
condition False


curretn None
parent human
data ['petya', {'names': ['barsik', 'mursik'], 'kind': 'animal', 'garbage': 123}]
condition False


curretn animal
parent human
data {'names': ['barsik', 'mursik'], 'kind': 'animal', 'garbage': 123}


TypeError: 'bool' object is not callable

In [56]:
def processingForStr(data: str, kind : str | None = None):
    return (kind is None and data) or None


def routergForDict(data: dict, kind : str | None = None):
     return (
         ("name" in data and processingForDict(data, kind) and (processingForDict(data, kind), True)) or 
         ("names" in data and processingForDicts(data, kind) and (processingForDicts(data, kind), True))
     )

# работает
def processingForDict(data: dict, kind : str | None = None):
    return (kind is None and data['name']) or ("kind" in data and kind is data["kind"] and data['name']) or None

# работает
def processingForDicts(data: dict, kind : str | None = None):
    return (kind is None and data['names']) or ("kind" in data and kind is data["kind"] and data['names']) or None


def processing(data: dict, kind: str | None = None, counter = None):
    counter = counter or 0

    is_str = type(data) is str
    is_int = type(data) is int
    is_dict = type(data) is dict
    is_list = type(data) is list

    key = is_dict and next(iter(data)) 

    current_kind = kind or (is_dict and 'kind' in data and data['kind']) or key

    print('Данные', data)
    print('ключ из объекта', key)
    print('итоговый ключ', current_kind)
    print('-----')

    return (
       (((is_dict and 'name' in data and (kind is None or not "kind" in data or data["kind"] == kind)) and processing(data['name'], current_kind)) or 
         ((is_dict and 'names' in data and (kind is None or not "kind" in data or data["kind"] == kind)) and processing(data['names'], current_kind)) or 
         ((is_str) and [data]) or []) 
        
        + 
        (is_list and counter < len(data) and (
            processing(data[counter], current_kind, 0) +
            processing(data, kind, counter + 1)
        ) or (
            is_dict and (key is not False) and processing(data[key], current_kind)
        ) or (
            is_dict and (key is False) and processing(data.values(), current_kind)
        ) or [])
    )





    # ) or (
    #     is_str and "name" in data and processingForStr(data["name"])
    # )

    
    
    # or (type(data[key]) is dict and key == "name" and (routergForDict(data, kind), True)) or (type(data[key]) is list and key == "name" and (routergForDict(data, kind), True)) or (type(data[key]) is list and key != "name" and (processingForCollection(data[key], kind), True))


def get_all_names(data: str | dict | list, kind: str | None = None) -> list[str]:


    
    """Collect all names from the `data` object. If `kind` is not None, only collects names of that kind.

    Arguments:
        data: object to collect names from

    Returns:
        Collected names
    """

    
    return (type(data) is str and processingForStr(data, kind)) or (type(data) is dict and routergForDict(data, kind)[0])

data2 = {
    'human': [
{
                'name': 'lyonya',
                'kind': 'animal'
            },
            {
                'name': 'vasya',
            },
            
        ]
} 

data3 = {
    'animal': [
{
                'name': 'ksenia',
                'kind': 'human'
            },
            {
                'name': 'lion',
            },
            
        ]
} 

data1 =  {
    'name': 'petya',
    'kind': 'human'
}

data = [
    123,
    {
        'names': ['barsik', 'mursik'],
        'kind': 'animal',
    },
    {
        'names': ['petya', 'vasya', 123],
        'kind': 'human',
    }
]


# print(get_all_names('fdsf'))

print(set(processing(data)))

Данные [123, {'names': ['barsik', 'mursik'], 'kind': 'animal'}, {'names': ['petya', 'vasya', 123], 'kind': 'human'}]
ключ из объекта False
итоговый ключ False
-----
Данные 123
ключ из объекта False
итоговый ключ False
-----
Данные [123, {'names': ['barsik', 'mursik'], 'kind': 'animal'}, {'names': ['petya', 'vasya', 123], 'kind': 'human'}]
ключ из объекта False
итоговый ключ False
-----
Данные {'names': ['barsik', 'mursik'], 'kind': 'animal'}
ключ из объекта names
итоговый ключ animal
-----
Данные ['barsik', 'mursik']
ключ из объекта False
итоговый ключ animal
-----
Данные barsik
ключ из объекта False
итоговый ключ animal
-----
Данные ['barsik', 'mursik']
ключ из объекта False
итоговый ключ animal
-----
Данные mursik
ключ из объекта False
итоговый ключ animal
-----
Данные ['barsik', 'mursik']
ключ из объекта False
итоговый ключ animal
-----
Данные [123, {'names': ['barsik', 'mursik'], 'kind': 'animal'}, {'names': ['petya', 'vasya', 123], 'kind': 'human'}]
ключ из объекта False
итоговый 

In [ ]:
def processingForDict(data: dict, key: str, kind : str | None = None):
    return (key in data and (kind is None or data.get("kind") == kind)) and data[key]


def processing(data: dict, kind: str | None = None, counter = None):
    counter = counter or 0

    is_str = type(data) is str
    is_dict = type(data) is dict
    is_list = type(data) is list

    key = (is_dict and next(iter(data))) or None

    current_kind = (is_dict and key not in ('name', 'names') and key) or kind or (is_dict and 'kind' in data and data['kind']) or None

    return (
        (
         (is_dict and processingForDict(data, "name", kind)) and processing(data['name'], current_kind) or
         (is_dict and processingForDict(data, "names", kind)) and processing(data['names'], current_kind) or
         (is_str and [data]) or []) 
        + 
        (is_list and counter < len(data) and (
            processing(data[counter], current_kind, 0) +
            processing(data, current_kind, counter + 1)
        ) or (
            is_dict and (key is not None) and (key not in ('name', 'names')) and processing(data[key], current_kind)
        ) or (
            is_dict and (key is None) and processing(list(data.values()), current_kind)
        ) or [])
    )


def get_all_names(data: str | dict | list, kind: str | None = None) -> list[str]:
    """Collect all names from the `data` object. If `kind` is not None, only collects names of that kind."""
    return processing(data, kind) or []


data = [
    'petya',
    {
        'names': ['barsik', 'mursik'],
        'kind': 'animal',
        'garbage': 123
    },
    {
        'human': [
            {
                'name': 'vasya',
            },
            {
                'name': 'lyonya',
                'kind': 'animal'
            }
        ]
    }
]

print(*get_all_names(data), ';')
print(*get_all_names(data, 'human'), ';')
print(*get_all_names(data, 'animal'), ';')